## Hybrid Recommender System

The goal of this experiment is to investigate whether combining two collaborative filtering approaches can improve recommendation performance on the MovieLens 100K dataset.

The hybrid recommender combines:

- **ItemKNN**, a neighborhood-based collaborative filtering method that recommends items based on item-item similarity.
- **SLIMElastic**, a sparse linear recommendation model that learns relationships between items using Elastic Net regularization.

The motivation for combining these models is that they capture item relationships in different ways. ItemKNN relies directly on similarity between items, while SLIMElastic learns sparse item-item coefficients from user interaction data. Combining their predictions may therefore provide complementary information and improve recommendation quality.

## Hybrid Method

The hybrid recommender combines the prediction scores produced by ItemKNN and SLIMElastic. Instead of manually assigning the contribution of each recommender, the weights are learned using a regression model.

For each user-item pair, the normalized prediction scores of ItemKNN and SLIMElastic are used as input features:

$$
X_{ui} =
\begin{bmatrix}
S_{\text{ItemKNN}}(u,i) &
S_{\text{SLIM}}(u,i)
\end{bmatrix}
$$

The target variable represents whether the item is relevant to the user:

$$
y_{ui} =
\begin{cases}
1 & \text{if item } i \text{ is relevant to user } u \\
0 & \text{otherwise}
\end{cases}
$$

A linear regression model is then fitted:

$$
\hat{y}_{ui}
=
\beta_0
+
\beta_1 S_{\text{ItemKNN}}(u,i)
+
\beta_2 S_{\text{SLIM}}(u,i)
$$

where $\beta_1$ and $\beta_2$ are learned from the validation data. The resulting predicted value $\hat{y}_{ui}$ is used as the final hybrid ranking score.

This approach allows the contribution of each individual recommender to be learned from data rather than manually specified.

### Evaluation Methodology

The validation set was used to learn the hybrid coefficients. The normalized
prediction scores from ItemKNN and SLIMElastic were used as input features
for a linear regression model.

For each user, relevant validation items were assigned a target value of 1,
while sampled non-relevant items were assigned a target value of 0.

The learned regression model is:

$$
\hat{y}_{ui}
=
\beta_0
+
\beta_1 S_{\mathrm{ItemKNN}}(u,i)
+
\beta_2 S_{\mathrm{SLIM}}(u,i)
$$

The learned coefficients determine the contribution of each recommender to
the final hybrid score. The test set was kept separate from the
weight-learning process and was used only for final evaluation.

Performance is reported using Recall@10, MRR@10, NDCG@10, Hit@10,
and Precision@10.

In [35]:
pip install --force-reinstall "numpy==1.23.5"

  Using cached numpy-1.23.5-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp310-cp310-macosx_11_0_arm64.whl (13.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
Note: you may need to restart the kernel to use updated packages.


In [36]:
import sys
import numpy as np
import torch
import recbole

# Should be NumPy  : 1.23.5, PyTorch: 2.0.1, RecBole: 1.2.1
print("Python :", sys.executable)
print("NumPy  :", np.__version__)
print("PyTorch:", torch.__version__)
print("RecBole:", recbole.__version__)

from recbole.quick_start import load_data_and_model
from recbole.utils.case_study import full_sort_topk

Python : /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/bin/python
NumPy  : 1.23.5
PyTorch: 2.0.1
RecBole: 1.2.1


In [37]:
from recbole.quick_start import run_recbole, load_data_and_model

itemknn_result = run_recbole(
    model="ItemKNN",
    dataset="ml-100k",
    config_dict={
        "k": 100,  # replace later with tuned value
    },
    saved=True
)

slim_result = run_recbole(
    model="SLIMElastic",
    dataset="ml-100k",
    config_dict={
        "alpha": 0.1,      # replace later with tuned value
        "l1_ratio": 0.1,   # replace later with tuned value
    },
    saved=True
)

07 Oct 22:37    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3e13b3e670cf1188bc22bf8c3b72aeec59cee77fa.json']
07 Oct 22:37    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 

In [38]:
from pathlib import Path

# Find newest saved checkpoint for each model
saved_dir = Path("saved")

ITEMKNN_CHECKPOINT = max(
    saved_dir.glob("ItemKNN-*.pth"),
    key=lambda p: p.stat().st_mtime
)

SLIM_CHECKPOINT = max(
    saved_dir.glob("SLIMElastic-*.pth"),
    key=lambda p: p.stat().st_mtime
)

print("Loading:")
print("ItemKNN:", ITEMKNN_CHECKPOINT)
print("SLIMElastic:", SLIM_CHECKPOINT)

# Load ItemKNN
(
    itemknn_config,
    itemknn_model,
    itemknn_dataset,
    itemknn_train,
    itemknn_valid,
    itemknn_test,
) = load_data_and_model(
    str(ITEMKNN_CHECKPOINT)
)

# Load SLIMElastic
(
    slim_config,
    slim_model,
    slim_dataset,
    slim_train,
    slim_valid,
    slim_test,
) = load_data_and_model(
    str(SLIM_CHECKPOINT)
)

itemknn_model.eval()
slim_model.eval()

print("\nModels loaded successfully.")

Loading:
ItemKNN: saved/ItemKNN-Oct-07-2026_22-37-43.pth
SLIMElastic: saved/SLIMElastic-Oct-07-2026_22-37-56.pth


07 Oct 22:37    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = False
metrics = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision']
topk = [10]
v


Models loaded successfully.


In [39]:
from sklearn.linear_model import LinearRegression

In [40]:
def normalize_rows(scores):
    scores = scores.clone()

    min_vals = scores.min(dim=1, keepdim=True).values
    max_vals = scores.max(dim=1, keepdim=True).values

    return (
        (scores - min_vals)
        / (max_vals - min_vals + 1e-12)
    )

In [41]:
X = []
y = []

itemknn_model.eval()
slim_model.eval()

rng = np.random.default_rng(42)

with torch.no_grad():

    for batch in itemknn_valid:

        (
            interaction,
            history_index,
            positive_u,
            positive_i,
        ) = batch

        interaction = interaction.to(
            itemknn_config["device"]
        )

        # Get scores from both recommenders
        item_scores = (
            itemknn_model
            .full_sort_predict(interaction)
            .view(len(interaction), -1)
        )

        slim_scores = (
            slim_model
            .full_sort_predict(interaction)
            .view(len(interaction), -1)
        )

        # Same normalization used by the hybrid
        item_scores = normalize_rows(item_scores)
        slim_scores = normalize_rows(slim_scores)

        # Relevant validation items for each user
        ground_truth = {}

        for u_idx, item_id in zip(
            positive_u.cpu().tolist(),
            positive_i.cpu().tolist()
        ):
            ground_truth.setdefault(
                int(u_idx),
                set()
            ).add(int(item_id))

        # Historical items should not be negative samples
        history = {}

        if history_index is not None:
            history_u, history_i = history_index

            for u_idx, item_id in zip(
                history_u.cpu().tolist(),
                history_i.cpu().tolist()
            ):
                history.setdefault(
                    int(u_idx),
                    set()
                ).add(int(item_id))

        num_items = item_scores.shape[1]

        for u_idx, relevant_items in ground_truth.items():

            # Positive examples
            for item_id in relevant_items:

                X.append([
                    item_scores[u_idx, item_id].item(),
                    slim_scores[u_idx, item_id].item(),
                ])

                y.append(1.0)

            # Candidate negatives:
            # not padding, history, or validation positives
            excluded = (
                history.get(u_idx, set())
                | relevant_items
                | {0}
            )

            candidates = np.array([
                item_id
                for item_id in range(1, num_items)
                if item_id not in excluded
            ])

            # Sample negatives
            n_negatives = min(
                10 * len(relevant_items),
                len(candidates)
            )

            if n_negatives > 0:
                negative_items = rng.choice(
                    candidates,
                    size=n_negatives,
                    replace=False
                )

                for item_id in negative_items:

                    X.append([
                        item_scores[u_idx, item_id].item(),
                        slim_scores[u_idx, item_id].item(),
                    ])

                    y.append(0.0)

X = np.asarray(X)
y = np.asarray(y)

print("Regression dataset")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive examples:", int(y.sum()))
print("Negative examples:", int((y == 0).sum()))

Regression dataset
X shape: (105556, 2)
y shape: (105556,)
Positive examples: 9596
Negative examples: 95960


In [42]:
import numpy as np
import torch
from recbole.data.interaction import Interaction

In [43]:
def metrics_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)

    hits = [
        1 if item in relevant else 0
        for item in recommended
    ]

    # Precision@K
    precision = sum(hits) / k

    # Recall@K
    recall = (
        sum(hits) / len(relevant)
        if relevant else 0.0
    )

    # Hit@K
    hit = 1.0 if sum(hits) > 0 else 0.0

    # MRR@K
    mrr = 0.0

    for rank, is_hit in enumerate(hits, start=1):
        if is_hit:
            mrr = 1.0 / rank
            break

    # NDCG@K
    dcg = sum(
        is_hit / np.log2(rank + 2)
        for rank, is_hit in enumerate(hits)
    )

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1.0 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "recall": recall,
        "mrr": mrr,
        "ndcg": ndcg,
        "hit": hit,
        "precision": precision,
    }

In [44]:
def evaluate_hybrid(eval_data, alpha=0.5, k=10):
    results = []

    itemknn_model.eval()
    slim_model.eval()

    with torch.no_grad():

        for batch in eval_data:

            (
                interaction,
                history_index,
                positive_u,
                positive_i,
            ) = batch

            interaction = interaction.to(
                itemknn_config["device"]
            )

            # --------------------------------
            # Predictions
            # --------------------------------

            item_scores = (
                itemknn_model
                .full_sort_predict(interaction)
            )

            slim_scores = (
                slim_model
                .full_sort_predict(interaction)
            )

            item_scores = item_scores.view(
                len(interaction),
                -1
            )

            slim_scores = slim_scores.view(
                len(interaction),
                -1
            )

            # --------------------------------
            # Normalize
            # --------------------------------

            item_scores = normalize_rows(
                item_scores
            )

            slim_scores = normalize_rows(
                slim_scores
            )

            # --------------------------------
            # Hybrid
            # --------------------------------

            scores = (
                (1 - alpha) * item_scores
                + alpha * slim_scores
            )

            # Padding item
            scores[:, 0] = -torch.inf

            # --------------------------------
            # Mask user's history
            # --------------------------------

            if history_index is not None:

                history_u, history_i = (
                    history_index
                )

                scores[
                    history_u,
                    history_i
                ] = -torch.inf

            # --------------------------------
            # Top-K
            # --------------------------------

            top_items = torch.topk(
                scores,
                k=k,
                dim=1
            ).indices.cpu()

            # --------------------------------
            # Ground truth
            # --------------------------------

            ground_truth = {}

            for u_idx, item_id in zip(
                positive_u.cpu().tolist(),
                positive_i.cpu().tolist()
            ):

                ground_truth.setdefault(
                    int(u_idx),
                    set()
                ).add(int(item_id))

            # --------------------------------
            # Metrics
            # --------------------------------

            for u_idx, relevant in ground_truth.items():

                recommended = (
                    top_items[u_idx]
                    .tolist()
                )

                result = metrics_at_k(
                    recommended,
                    relevant,
                    k=k
                )

                results.append(result)

    # --------------------------------
    # Average across users
    # --------------------------------

    final = {
        metric: np.mean([
            result[metric]
            for result in results
        ])
        for metric in [
            "recall",
            "mrr",
            "ndcg",
            "hit",
            "precision",
        ]
    }

    return final

In [45]:
itemknn_check = evaluate_hybrid(
    itemknn_test,
    alpha=0.0,
    k=10
)

print("ItemKNN sanity check")

for metric, value in itemknn_check.items():
    print(f"{metric}@10: {value:.4f}")

ItemKNN sanity check
recall@10: 0.2470
mrr@10: 0.4623
ndcg@10: 0.2834
hit@10: 0.7847
precision@10: 0.1931


In [46]:
slim_test = evaluate_hybrid(
    itemknn_test,
    alpha=1.0,
    k=10
)

print("SLIMElastic Test Results")

for metric, value in slim_test.items():
    print(f"{metric}@10: {value:.4f}")

SLIMElastic Test Results
recall@10: 0.2646
mrr@10: 0.5113
ndcg@10: 0.3108
hit@10: 0.8038
precision@10: 0.2071


### Learning the Hybrid Weights

The normalized ItemKNN and SLIMElastic scores are used as input features for a linear regression model. The target indicates whether an item is relevant to the user.

The regression model learns the contribution of each recommender:

$$
\hat{y}_{ui}
=
\beta_0
+
\beta_1 S_{\text{ItemKNN}}(u,i)
+
\beta_2 S_{\text{SLIM}}(u,i)
$$

In [47]:
def evaluate_regression_hybrid(
    eval_data,
    regression,
    k=10
):
    results = []

    itemknn_model.eval()
    slim_model.eval()

    # Learned regression parameters
    intercept = float(regression.intercept_)
    itemknn_coef = float(regression.coef_[0])
    slim_coef = float(regression.coef_[1])

    with torch.no_grad():

        for batch in eval_data:

            (
                interaction,
                history_index,
                positive_u,
                positive_i,
            ) = batch

            interaction = interaction.to(
                itemknn_config["device"]
            )

            # ItemKNN scores
            item_scores = (
                itemknn_model
                .full_sort_predict(interaction)
                .view(len(interaction), -1)
            )

            # SLIM scores
            slim_scores = (
                slim_model
                .full_sort_predict(interaction)
                .view(len(interaction), -1)
            )

            # Normalize scores
            item_scores = normalize_rows(item_scores)
            slim_scores = normalize_rows(slim_scores)

            # Regression-based weighted hybrid
            scores = (
                intercept
                + itemknn_coef * item_scores
                + slim_coef * slim_scores
            )

            # Exclude padding item
            scores[:, 0] = -torch.inf

            # Exclude previously interacted items
            if history_index is not None:
                history_u, history_i = history_index
                scores[history_u, history_i] = -torch.inf

            # Top-K recommendations
            top_items = torch.topk(
                scores,
                k=k,
                dim=1
            ).indices.cpu()

            # Ground-truth relevant items
            ground_truth = {}

            for u_idx, item_id in zip(
                positive_u.cpu().tolist(),
                positive_i.cpu().tolist()
            ):
                ground_truth.setdefault(
                    int(u_idx),
                    set()
                ).add(int(item_id))

            # Calculate metrics
            for u_idx, relevant in ground_truth.items():

                recommended = top_items[u_idx].tolist()

                result = metrics_at_k(
                    recommended,
                    relevant,
                    k=k
                )

                results.append(result)

    return {
        metric: np.mean([
            result[metric]
            for result in results
        ])
        for metric in [
            "recall",
            "mrr",
            "ndcg",
            "hit",
            "precision",
        ]
    }

In [50]:
from sklearn.linear_model import LinearRegression

regression = LinearRegression()

regression.fit(X, y)

intercept = float(regression.intercept_)
itemknn_coef = float(regression.coef_[0])
slim_coef = float(regression.coef_[1])

print("Learned regression model")
print(f"Intercept:           {intercept:.4f}")
print(f"ItemKNN coefficient: {itemknn_coef:.4f}")
print(f"SLIM coefficient:    {slim_coef:.4f}")

Learned regression model
Intercept:           -0.0086
ItemKNN coefficient: 0.2395
SLIM coefficient:    0.7620


### Learned Hybrid Weights

The linear regression model learned the following coefficients:

- **ItemKNN:** 0.2395
- **SLIMElastic:** 0.7620
- **Intercept:** -0.0086

The coefficients of the two recommenders sum to approximately 1.0, meaning
that the learned hybrid assigns roughly 24% of the contribution to ItemKNN
and 76% to SLIMElastic.

This indicates that SLIMElastic contributes more strongly to the final
ranking, while ItemKNN still provides a complementary signal. The weights
were learned from the validation data rather than manually selected.

In [51]:
regression_test = evaluate_regression_hybrid(
    itemknn_test,
    regression=regression,
    k=10
)

print("Regression Weighted Hybrid — Test Results")

for metric, value in regression_test.items():
    print(f"{metric.upper()}@10: {value:.4f}")

Regression Weighted Hybrid — Test Results
RECALL@10: 0.2704
MRR@10: 0.5215
NDCG@10: 0.3188
HIT@10: 0.8070
PRECISION@10: 0.2124
